# Working with numerical data - Part I

This notebook accompanies the script <strong><span style="color:red;">03_Numpy_partA.pdf</span></strong>  and provides practical examples related to its content.

In [ ]:
import numpy as np

In [ ]:
print(f"NumPy version {np.__version__}") # Dunder Attribut / Magic Attribut

<hr style="border: none; height: 20px; background-color: green;">

# 1. Introduction to working with NumPy arrays

Real-world data must be transformed into numerical form to enable efficient computation.

NumPy provides a unified structure for such data.

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Example: Audio

### Goose WAV file into NumPy array

In [ ]:
from scipy.io import wavfile
from IPython.display import Audio
import matplotlib.pyplot as plt

# use scipy.io to load WAV into NumPy array
with open("../data/audio/gooose.wav", "rb") as f:
    rate, data = wavfile.read(f)
    
print("Sampling rate (Hz):", rate)
print("Data shape (samples × channels):", data.shape)
print("Data type (Python):", type(data))
print("Data type (NumPy dtype):", data.dtype)

Audio(data, rate=rate)

#### Quick visual overview of the audio signal using matplotlib

In [ ]:
time_axis = np.arange(len(data)) / rate

plt.figure(figsize=(12, 4)) 
plt.plot(time_axis, data)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Audio Waveform")
plt.show()

#### Waveform close‑up: resolving discrete sample points

In [ ]:
# Time window
start_s = 5.0
duration_s = 0.01

# Convert to samples
start = int(start_s * rate)
end = start + int(rate * duration_s)

# Slice
data_part = data[start:end]

# Time axis
time_axis = np.arange(len(data_part)) / rate + start_s

# Plot
plt.figure(figsize=(12, 4)) 
plt.plot(time_axis, data_part, marker=".")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title(f"Audio Waveform ({start_s} - {start_s+duration_s} s)")
plt.grid(True, axis="y")
plt.show()

#### Frequency Spectrum of the Audio Signal

This code converts the audio signal from the **time domain** into the **frequency domain** using an FFT.  

The waveform shows how the amplitude changes over time, while the spectrum reveals which frequencies are present and how strong they are. This is useful for understanding the tonal structure of a sound, identifying dominant frequency components, and analyzing characteristics that are not visible in the raw waveform.

In [ ]:
# Parameters
start_s = 2.0
duration_s = 0.02  # 20 ms

# Convert to samples
start = int(start_s * rate)
end = start + int(duration_s * rate)

# Slice
segment = data[start:end].astype(float)

# Remove DC offset
segment -= segment.mean()

# Windowing
window = np.hanning(len(segment))
segment_win = segment * window

# FFT
fft = np.fft.rfft(segment_win)
freq = np.fft.rfftfreq(len(segment_win), d=1/rate)
magnitude = np.abs(fft)

# Plot
plt.figure(figsize=(12, 4))
plt.plot(freq, magnitude)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.title(f"Short-Time Spectrum ({start_s:.2f}–{start_s+duration_s:.2f} s)")
plt.grid(True, axis="y")
plt.show()


<hr style="border: none; height: 10px; background-color: LightBlue;">

## Example: ECG – Real Biomedical Signals

In [ ]:
# Library for reading WaveForm Database (WFDB) files such as .hea and .dat
import wfdb

# The loaded WFDB Record contains both metadata and the signal stored as a NumPy array
record = wfdb.rdrecord('../data/ecg_db/patient001/s0010_re')

# The ECG signal is stored inside the record as a NumPy array (p_signal).
print(type(record.p_signal))

# Access the raw signal values (all channels) as a NumPy array. 
# Here we show the first three samples to verify the structure.
record.p_signal[0:3]

### Selecting and plotting specific ECG leads

This section extracts a subset of ECG channels from the already‑loaded WFDB record and prepares them for visualization. 

The signal matrix is first transposed so that each row corresponds to one lead. A list of standard 12‑lead ECG names is used to identify the indices of the desired channels. These channels are then selected and stored together with their corresponding lead names. A time window is defined in samples and converted into seconds using the sampling rate. 

Finally, only the inferior leads (II, III, aVF) are plotted over the chosen time interval to show their waveform morphology within that segment.

In [ ]:
# Transpose: (channels, samples)
channels = record.p_signal.T

selected_leads = [
    "i", "ii", "iii", "avr", "avl", "avf",
    "v1", "v2", "v3", "v4", "v5", "v6"
]

# Find indices of selected leads
lead_indices = [
    i for i, name in enumerate(record.sig_name)
    if name.lower() in selected_leads
]

# Extract selected channels
channels_12 = channels[lead_indices]

# Corresponding lead names
lead_names = [record.sig_name[i] for i in lead_indices]

# Define time window (in samples)
start_idx = 1300
end_idx   = 2600

fs = record.fs  # sampling rate (Hz)

# Time axis in seconds
t = np.arange(start_idx, end_idx) / fs

# Define lead groups
inferior_leads = ["ii", "iii", "avf"]

# Plot inferior leads
plt.figure(figsize=(15, 10))
fontsize = 20

for name, signal in zip(lead_names, channels_12):
    # Plot only inferior leads
    if name.lower() in inferior_leads:

        plt.plot(
            t,
            signal[start_idx:end_idx],
            label=name.upper()
        )

plt.xlabel("Time (s)", fontsize=fontsize)
plt.ylabel("Amplitude (mV)", fontsize=fontsize)
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.title(
    "ECG Signal - Inferior Leads – Patient 001",
    fontsize=28
)
plt.legend(
    loc="upper right",
    fontsize=fontsize - 4,
    title="Leads",
    title_fontsize=fontsize - 2
)
plt.grid(True, alpha=0.3)
plt.tight_layout()

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Example: Image - Cat

In [ ]:
from PIL import Image

# Load an image file (JPEG) using PIL
img = Image.open("../data/images/hugo.jpg")

# Convert the image into a NumPy array (height × width × color channels)
arr = np.array(img)

# Check that the result is a NumPy array
print(type(arr))

# Show the array shape: (rows, columns, channels)
arr.shape

In [ ]:
plt.imshow(arr)
plt.axis("off");

<hr style="border: none; height: 20px; background-color: LightBlue;">

## Numpy Basics: Creating NumPy Arrays

### From Python Lists

Creating a NumPy array from a python list `[1, 2, 3, 4, 5]`

In [ ]:
# integer array
arr = np.array([1, 2, 3, 4, 5])
arr

In [ ]:
# numpy.ndarray (n-dimensional array)
type(arr)

### Explicit Data Type

We can optionally set the data type explicitly

In [ ]:
# float array
arr_float = np.array([1, 2, 3, 4, 5], dtype=np.float32)
arr_float

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Array Attributes

Every NumPy array exposes metadata that describes its structure.



| Attribute  | Meaning                          |
|------------|----------------------------------|
| dtype      | Data type of elements            |
| ndim       | Number of dimensions             |
| shape      | Size per dimension               |
| strides    | Memory step size                 |
| size       | Total number of elements         |
| itemsize   | Bytes per element                |


In [ ]:
arr = np.array([[0,1,2],
                [3,4,5],
                [6,7,8]], 
               dtype=np.float32)

print(arr)

print("dtype:", arr.dtype)
print("ndim:", arr.ndim)
print("shape:", arr.shape)
print("strides:", arr.strides)
print("size:", arr.size)
print("itemsize:", arr.itemsize)

<hr style="border: none; height: 10px; background-color: LightBlue;">

### Arrays with Zeros or Ones

We can create new arrays initialized with zeros or ones.
The **first argument** defines the **shape**, and `dtype` specifies the data type.

In [ ]:
# create a length-10 integer array filled with zeros
np.zeros(10, dtype=int)

In [ ]:
# create a 3x5 floating-point array filled with ones
np.ones((3,5), dtype=float)

### Array with a Specific Constant Value

We can also create new arrays that are filled with a given constant value.

In [ ]:
# create a 3x5 array filled with 3.14
np.full((3,5), 3.14)

### Ranges

With `np.arange()`, we can generate evenly spaced values within a given interval.  
It works similarly to Python’s built-in `range()`, but returns a NumPy array instead of a range object.

#### Three Common Call Forms of `np.arange()`

In [ ]:
stop = 10
start = 2
step = 2

# Generates values from 0 up to, but not including, stop
print(np.arange(stop))

# Generates values from start up to, but not including, stop
print(np.arange(start, stop))

# Generates values from start to stop using increments of step
print(np.arange(start, stop, step))

# Explicitly specifying the data type
print(np.arange(start, stop, step, dtype=float))

#### More examples of `np.arange`: 

Steps, Floats, Descending Sequences and Reshaping

In [ ]:
# Creating an array with floating-point numbers and float steps
arr_float = np.arange(0, 1, 0.2, dtype=np.float32)
print("Array with float step 0.2:", arr_float)

In [ ]:
# Creating a descending array
arr_desc = np.arange(10, 0, -2)
print("Descending array:", arr_desc)

In [ ]:
# Combining np.arange() with reshape()
arr = np.arange(16).reshape(4, 4)
print("2dim array:", arr)

#### Introduction to `reshape()`

`reshape()` changes the shape of an array without changing its data.

The **total number of elements must remain the same.**   
Only the arrangement of the elements is modified.

In [ ]:
# Create a 1D array with 12 elements
arr = np.arange(12)
print(arr.shape)
print(arr)

In [ ]:
# Rearrange the same elements into a 3×4 2D array.
reshaped = arr.reshape(3, 4)
print(reshaped.shape)
print(reshaped)

Automatically infer one dimension:

In [ ]:
# You can use `-1` to let NumPy automatically infer one dimension.
reshaped_auto = arr.reshape(3,-1)
print("Reshaped with -1:\n", reshaped_auto.shape)
print(reshaped_auto)

#### Creating a Random NumPy Array

`np.random` lets you generate arrays filled with random numbers, covering uniform values, normal distributions, integers, and more.

- Uniform random values with `np.random.random()` or `np.random.rand()`
- Random integers with `np.random.randint()`
- Normally distributed values with `np.random.randn()` or `np.random.normal()`
- Random choices from lists or arrays with `np.random.choice()`
- Shuffling and permutations with `np.random.shuffle()` and `np.random.permutation()`

In [ ]:
arr = np.random.random((2,3,4))
print(arr)
arr.shape

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Data Types (dtype)

Choosing the correct dtype is important for:
- Memory efficiency
- Numerical precision
- Performance

#### When constructing an array, we can specify the dtype in different ways:

**Recommendation:** Prefer NumPy dtype objects for clarity, type safety, and platform-independent behavior.

In [ ]:
np.zeros(10, dtype=int)        # Python built-in type (platform-dependent)
np.zeros(10, dtype='int16')    # String alias
np.zeros(10, dtype=np.int16)   # NumPy dtype object (explicit)

#### Different dtypes use different amounts of memory

In [ ]:
import sys

arr_int8 = np.zeros(1_000_000, dtype=np.int8)
arr_int64 = np.zeros(1_000_000, dtype=np.int64)

# Memory used by the data buffer only
print("int8  data bytes: ", arr_int8.nbytes)
print("int64 data bytes: ", arr_int64.nbytes)

# Memory of the NumPy array object (data + metadata + object overhead)
print("int8  total object byte size: ", sys.getsizeof(arr_int8))
print("int64 total object byte size: ", sys.getsizeof(arr_int64))

#### Precision Difference

In [ ]:
# Same number stored with different precision
arr_float32 = np.array([1.1234567890123456789], dtype=np.float32)
arr_float64 = np.array([1.1234567890123456789], dtype=np.float64)

print("Default print:")
print("float32:", arr_float32)
print("float64:", arr_float64)

In [ ]:
# Increase print precision
np.set_printoptions(precision=15)

print("\nHigher print precision:")
print("float32:", arr_float32)
print("float64:", arr_float64)

diff = arr_float64[0] - arr_float32[0]

print("\nScientific notation difference:")
print("Difference:", diff)

print("\nFixed-point difference:")
print(f"Difference (fixed): {diff:.15f}")

# Reset print options to defaults
np.set_printoptions()

#### Compare element-wise computation for float32 and float64.

Smaller data types require less memory and can improve computational efficiency, especially for large arrays.   
As a result, `float32` operations are often slightly faster than `float64`, depending on the hardware and NumPy build.

In [ ]:
arr_float32 = np.random.random(10_000_000).astype(np.float32)
arr_float64 = np.random.random(10_000_000).astype(np.float64)

%timeit np.square(arr_float32)
%timeit np.square(arr_float64)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Type Casting and Upcasting

NumPy handles mixed types differently depending on the operation.

While Python lists can contain multiple data types.  
NumPy arrays contain only one type.

If types do not match, NumPy will promote if possible.

### Integers are upcasted to floating point:

In [ ]:
# Integers are upcasted to floating point:
arr = np.array([3.14, 4, 2, 3])
print(arr, arr.dtype)

### Arrays can only contain one data type

We cannot mix different numerical data types in the same array.

In [ ]:
# Create a 2D NumPy array
arr = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])
print(arr, arr.dtype)

If we assign a float value to an integer array element, it will be truncated toward zero and stored as an integer.

In [ ]:
# Modify one element
arr[1, 0] = 9.9
print(arr, arr.dtype)

In [ ]:
# Modify one element
arr[1, 0] = -1.9
print(arr, arr.dtype)

#### Result Stored in a New Array

In [ ]:
arr = np.array([1, 2])
print(id(arr))
arr = arr + 1.9   # Creates an new array with float values
print(id(arr))

#### In-place operation - Adding `float` to `int` array


In [ ]:
arr = np.array([1, 2])
print(id(arr))

try:
    arr += 1.9 # changes values in-place
except TypeError as e:
    print("In-place addition failed due to dtype casting. Converting array to float.")
    print("Original error:", e)

    arr = arr.astype(float)
    print(id(arr))
    arr += 1.9
    print(id(arr))

arr

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Indexing

### 1D Indexing


In [ ]:
arr = np.array([5, 0, 3, 3, 7, 9])
print(arr)

In [ ]:
arr[0] # Indexing from the beginning of the array

In [ ]:
arr[-1] # Indexing from the end of the array

### 2D Indexing

In [ ]:
arr = np.arange(16).reshape(4, 4)
print(arr)

#### Examples

In [ ]:
print(arr[1,0])

In [ ]:
print(arr[2,-1])

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Iteration

### Define a NumPy Array

In [ ]:
arr = np.array([
    [0.3, 0.4, 0.8],
    [0.5, 0.6, 0.7]
])
print(arr)

### Explicit Row-Column Iteration in a NumPy Array

In [ ]:
rows, cols = arr.shape

for i in range(rows):
    for j in range(cols):
        print(i, j, arr[i, j])

### Using shape to get Rows and Columns

In [ ]:
# Get rows and columns using shape
print(f"shape: {arr.shape}")  # (2, 3)

rows, columns = arr.shape
print(f"Rows: {rows}, Columns: {columns}")

### Iteration: Cleaner and more Pythonic with `enumerate()`

A cleaner and more Pythonic approach to iteration is to use `enumerate()`,
which eliminates the need for manual indexing, improving both readability and efficiency


In [ ]:
for i, row in enumerate(arr):          # i = row index, row = full row
    for j, value in enumerate(row):    # j = column index, value = element
        print(i, j, value)

### Iteration using `np.ndenumerate`

NumPy provides its own optimized iteration method: `np.ndenumerate()`.  
This approach is more compact, works seamlessly with arrays of any dimension and is particularly efficient for NumPy arrays.   
By using NumPy’s built-in optimizations, it allows for faster and more readable iteration compared to traditional looping methods.

In [ ]:
for (i, j), value in np.ndenumerate(arr):
    print(i, j, value)

<hr style="border: none; height: 20px; background-color: green;">

# 2. Slicing and copying

Slicing allows you to extract specific parts of a NumPy array using the `[start:stop:step]` syntax  

```
[start:stop:step]
```


In [ ]:
arr = np.arange(10)
print(arr)

In [ ]:
arr[:5] # First five elements

In [ ]:
arr[5:] # Elements after index 5

In [ ]:
arr[4:7] # Middle sub-array (index 4 to 6)

In [ ]:
arr[::2] # Every other element

In [ ]:
arr[1::2] # Every other element, starting at index 1

### Array Slicing: Extracting Subarrays from Multi-Dimensional Arrays

Create a 5×5 NumPy array with increasing integers:

```
array([[ 0,  1,  2,  3,  4,  5],
       [10, 11, 12, 13, 14, 15],
       [20, 21, 22, 23, 24, 25],
       [30, 31, 32, 33, 34, 35],
       [40, 41, 42, 43, 44, 45],
       [50, 51, 52, 53, 54, 55]])
```

In [ ]:
# Option 1: Create a 6x6 array using broadcasting (row vector + column vector)
arr = np.arange(6) + np.arange(0, 60, 10).reshape(6, 1)

# Option 2: Create the same 6x6 array explicitly using nested list comprehension
arr = np.array([[row * 10 + col for col in range(6)] for row in range(6)])

print(arr)

In [ ]:
arr[0,3:5]

In [ ]:
arr[4:,4:]

In [ ]:
arr[:,2]

In [ ]:
arr[2::2,::2]

### What Happens When We Assign a Slice to a New Variable?

In [ ]:
arr = np.array([10, 20, 30, 40, 50])
print("arr before:", arr)

# Slicing creates a view
arr_view = arr[1:4]

# Modifying the slice
arr_view[0] = 999

print("view:      ", arr_view)
print("arr after: ", arr)

### Creating Independent Arrays with `.copy()`

In [ ]:
arr = np.array([10, 20, 30, 40, 50])
print("arr before:", arr)

# Explicit copy
arr_copy = arr[1:4].copy()

# Modifying the copy
arr_copy[0] = 999

print("copy:      ", arr_copy)
print("arr after: ", arr)

#### Views vs Copies in 2D Array Slicing
This example shows the difference between creating a view and creating an explicit copy in a 2D NumPy array.


In [ ]:
arr = np.array([
    [10, 11, 12],
    [20, 21, 22],
    [30, 31, 32],
    [40, 41, 42]
])

In [ ]:
# Creating a View (Slicing without Copy)
arr_view = arr[:3, :]
print("array_view:\n", arr_view)

In [ ]:
arr_view[2, 2] = 999   # was 32

print("arr_view:\n", arr_view)
print("arr:\n", arr)   # original is modified

In [ ]:
arr = np.array([
    [10, 11, 12],
    [20, 21, 22],
    [30, 31, 32],
    [40, 41, 42]
])

In [ ]:
# Creating a Copy (Explicit Copy)
arr_copy = arr[:3, :].copy()

In [ ]:
arr_copy[2, 2] = 999   # was 32

print("\nAfter modifying the copy:")
print("arr_copy:\n", arr_copy)
print("arr:\n", arr)   # original remains unchanged

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Memory Ownership (.base)

The `.base` attribute shows the original owner of the data.


In [ ]:
arr = np.arange(10)
view = arr[::2]

print("arr.base:", arr.base)
print("arr owns memory:", arr.base is None)
print("\nview.base:", view.base)
print("view shares memory with arr:", view.base is arr)

#### Copy of an array: base

In [ ]:
arr_copy = arr.copy()
arr_copy.base is arr

In [ ]:
arr_copy = arr[::2].copy()

print("Copy.base:", arr_copy.base)
print("Copy shares memory with arr:", arr_copy.base is arr)

<hr style="border: none; height: 10px; background-color: LightBlue;">

### Understanding `deepcopy()` in NumPy

If your NumPy array contains nested objects (e.g., lists, dictionaries), use `copy.deepcopy()` instead of `.copy()` to avoid unexpected modifications

In [ ]:
import copy

# NumPy array containing Python lists (dtype=object)
arr = np.array([[1, [2, 3]],
                [4, [5, 6]]], dtype=object)

# Shallow copy (duplicates array, shares inner objects)
shallow = arr.copy()

# Deep copy (duplicates array and inner objects)
deep = copy.deepcopy(arr)

print("Before modification:")
print("arr:\n", arr)
print("shallow:\n", shallow)
print("deep:\n", deep)

# Modify an element inside a nested list
shallow[0, 1][0] = 99

print("\nAfter modifying shallow[0, 1][0] = 99:")
print("arr:\n", arr)        # Modified
print("shallow:\n", shallow)  # Modified
print("deep:\n", deep)      # Unchanged

<hr style="border: none; height: 10px; background-color: LightBlue;">

#### Transposing Arrays with `.T`

`.T` returns the transposed view of an array.

- For 1D arrays, nothing changes.
- For 2D arrays, rows and columns are swapped.
- For 3D (and higher) arrays, the order of all axes is reversed.

Use `.transpose()` if you want to swap specific axes instead of reversing all of them.

In [ ]:
# 1D Example - For 1D arrays, .T has no effect.
arr_1d = np.array([1, 2, 3, 4])
print("Original shape:", arr_1d.shape)

print("arr_1d.T:", arr_1d.T)
print("Transposed shape:", arr_1d.T.shape)

In [ ]:
# 2D Example
arr_2d = np.array([[1, 2, 3],
                   [4, 5, 6]])

print("Original shape:", arr_2d.shape)
print(arr_2d)

print("\nTransposed:")
print(arr_2d.T)
print("Transposed shape:", arr_2d.T.shape)

In [ ]:
# 3D Example
arr_3d = np.arange(24).reshape(2, 3, 4)

print("Original shape:", arr_3d.shape)

arr_3d_T = arr_3d.T
print("Transposed shape:", arr_3d_T.shape)

arr_swap = arr_3d.transpose(1, 0, 2)
print("Swapped shape (1,0,2):", arr_swap.shape)

<hr style="border: none; height: 20px; background-color: green;">

# 3. Concatenating and splitting numpy arrays

## 14. Concatenation

Combine arrays along existing axes.


In [ ]:
arr_1 = np.array([0.23, 0.73, 0.38])
arr_2 = np.array([0.82, 0.12, 0.95])

np.concatenate([arr_1, arr_2])

In [ ]:
arr_3 = np.array([0.41, 0.66])
np.concatenate([arr_1, arr_2, arr_3])

#### Concatenating Two-Dimensional Arrays

In [ ]:
arr_1 = np.array([[1, 2, 3], [4, 5, 6]])
arr_2 = np.array([[7, 8, 9]])

arr = np.concatenate([arr_1, arr_2])
print(arr)

In [ ]:
arr_1 = np.array([[1, 2], [3, 4], [5, 6]])
arr_2 = np.array([[5, 6, 7], [8, 9, 10], [11, 12, 13]])

arr_3 = np.concatenate([arr_1, arr_2], axis=1)
print(arr_3)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Splitting

Split arrays into multiple parts.

In [ ]:
# Create a 6x6 array using broadcasting
arr = np.arange(6) + np.arange(0, 60, 10).reshape(6, 1)
print("Original array:\n", arr)

#### Vertical splitting (split along rows → axis=0)


In [ ]:
arr_1, arr_2, arr_3 = np.split(arr, 3, axis=0)
print("\nVertical split using split(axis=0):")
print(arr_1, arr_2, arr_3, sep="\n\n")

#### Horizontal splitting (split along columns → axis=1)


In [ ]:
arr_1, arr_2, arr_3 = np.split(arr, 3, axis=1)
print("\nHorizontal split using split(axis=1):")
print(arr_1, arr_2, arr_3, sep="\n\n")

<hr style="border: none; height: 20px; background-color: green;">

# 4. Vectorized Operations: The Heart of NumPy Performance


## Vectorization and Performance

Vectorized operations are implemented in optimized C code.
They are significantly faster than Python loops.

#### Example: Computing square root manually



In [ ]:
import math

# Generate random floats
arr = np.random.random(1_000_000)

def sqrt_loop(values):
    output = np.empty(len(arr))  # Creates an uninitialized array
    for i in range(len(arr)):
        output[i] = math.sqrt(arr[i])
    return output

# Measure execution time with loop
%timeit sqrt_loop(arr)

#### Example: Using NumPy’s UFuncs

In [ ]:
# Using NumPy' build-in vectorized function
%timeit np.sqrt(arr)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Unary and Binary UFuncs: NumPy’s Element-Wise Operators

- NumPy provides two main types of ufuncs: unary and binary
- Unary ufuncs operate on a single input array element-wise, such as `np.sqrt()`
- Binary ufuncs take two input arrays and perform element-wise operations, like `np.add()`

### Element‑wise results of common NumPy ufuncs.

In [ ]:
arr_1 = np.array([1, 7, 6, 4])
arr_2 = np.array([5, 2, 8, 3])

# Unary ufuncs
print("np.abs(arr_1)  :", np.abs(arr_1))
print("np.exp(arr_1)  :", np.round(np.exp(arr_1), 3))
print("np.sqrt(arr_1) :", np.round(np.sqrt(arr_1), 3))

# Binary ufuncs
print("np.add(arr_1, arr_2)     :", np.add(arr_1, arr_2))
print("np.multiply(arr_1, arr_2):", np.multiply(arr_1, arr_2))

# Multi‑input ufunc
print("np.where(arr_1 > 4, arr_2, 0):", np.where(arr_1 > 4, arr_2, 0))


<hr style="border: none; height: 10px; background-color: LightBlue;">

#### UFuncs and `NaN` propagation 
UFuncs apply operations elementwise and propagate `NaN` values, so any computation involving `NaN` produces `NaN` and comparisons with `NaN` are always false.

In [ ]:
arr = np.array([1.0, np.nan, 4.0])

print(np.sqrt(arr))        # [1. nan 2.]
print(np.add(arr, 10))     # [11. nan 14.]
print(np.equal(arr, arr))  # [ True False  True ]

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Using the out Parameter

When working with NumPy ufuncs, specifying an output array via the out parameter can improve efficiency by avoiding additional memory allocations.  

Note: Python lists do not support an equivalent mechanism.

In [ ]:
arr = np.arange(1_000_000, dtype=float)
arr_out = np.empty_like(arr)          # preallocated output array
np.sqrt(arr, out=arr_out)             # write results directly into y

print(arr_out)

`np.empty()` allocates an array without initializing its contents, so the printed values are arbitrary memory data.

In [ ]:
arr = np.empty(8, dtype=np.float64)   # uninitialized memory
print(arr)                            # contains arbitrary values

<hr style="border: none; height: 20px; background-color: green;">

# 5 Aggregations: mean; std; min; max; 

## Aggregations

The plot shows the probability density of daily average temperatures in 2024 for Basel and Jungfraujoch.   
Basel exhibits a much warmer distribution with higher mean temperatures, while Jungfraujoch is centered around sub-zero values, reflecting its high-altitude alpine climate.  

In [ ]:
from scipy.stats import gaussian_kde

# Load tavg column (index 1)
basel = np.genfromtxt(
    "../data/csv/basel_2024.csv", delimiter=",", skip_header=1, usecols=1
)
jung = np.genfromtxt(
    "../data/csv/jungfraujoch_2024.csv", delimiter=",", skip_header=1, usecols=1
)

# Remove NaNs
basel = basel[~np.isnan(basel)]
jung = jung[~np.isnan(jung)]

# KDE
kde_basel = gaussian_kde(basel)
kde_jung = gaussian_kde(jung)

x_values = np.linspace(-30, 35, 500)

# Plot
plt.figure(figsize=(12, 7))
plt.plot(x_values, kde_basel(x_values), label="Basel")
plt.plot(x_values, kde_jung(x_values), label="Jungfraujoch")

# Mean lines
plt.axvline(basel.mean(), linestyle="--",
            label=f"Mean Basel: {basel.mean():.1f}°C")
plt.axvline(jung.mean(), linestyle="--",
            label=f"Mean Jungfraujoch: {jung.mean():.1f}°C")

plt.title("Temperature Distribution in 2024: Basel vs. Jungfraujoch")
plt.xlabel("Temperature (°C)")
plt.ylabel("Density")
plt.legend()
plt.grid(True)

plt.show()


<hr style="border: none; height: 10px; background-color: LightBlue;">

## An Example with summing values of an array

We create a random array of 1000 items

In [ ]:
arr = np.random.random(1_000)
big_arr = np.random.random(1_000_000)

We can compute the sum of all items of the array with:
- built-in python `sum()` function
- numpy `np.sum()`

In [ ]:
# small array
%timeit sum(arr)
%timeit np.sum(arr)

In [ ]:
# bit array
%timeit sum(big_arr)
%timeit np.sum(big_arr)

#### Differences between `sum()` and `np.sum()`

In [ ]:
np.random.seed(42) # Set a seed for reproducible random numbers

arr = np.random.random(1_000_000)

# Compare Python's sum() and NumPy's np.sum()
python_sum = sum(arr)
numpy_sum  = np.sum(arr)

print("sum(arr)   :", python_sum)
print("np.sum(arr):", numpy_sum)
print("Difference :", python_sum - numpy_sum)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Operations on multidimensional arrays

In [ ]:
arr = np.random.random((3, 4))
print(arr)

In [ ]:
arr.sum(axis=1)

In [ ]:
arr.sum(axis=0)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## NaN-Safe Aggregations

Creating **Example Data** to play with aggregating datasets

In [ ]:
# Create a random number generator with a fixed seed (reproducible results)
rng = np.random.default_rng(42)

# Generate a 2D dataset with shape (100, 5)
# 100 rows (samples), 5 columns (features)
# Values are drawn from a normal distribution:
# mean (loc) = 10, standard deviation (scale) = 3
arr = rng.normal(loc=10, scale=3, size=(100, 5))

# Create a boolean mask with the same shape as arr
# About 5% of the entries will be True
mask = rng.random(arr.shape) < 0.05

# Replace selected entries with NaN (simulate missing values)
arr[mask] = np.nan

# Print first 4 rows rounded to 3 decimal places
print(np.round(arr[:4], 3))

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Basic Reduction Operations in NumPy

### Mean, Standard Deviation, Variance

In [ ]:
mean = np.mean(arr)
std = np.std(arr)
var = np.var(arr)

print(f"Mean:               {mean}")
print(f"Standard deviation: {std}")
print(f"Variance:           {var}")

<hr style="border: none; height: 10px; background-color: LightBlue;">

## NaN-Safe Reduction Operations

### Handling Missing Values

NumPy provides nan* variants that ignore NaN values.

In [ ]:
mean_nan = np.nanmean(arr)
std_nan = np.nanstd(arr)
var_nan = np.nanvar(arr)

print(f"Mean:               {mean_nan}")
print(f"Standard deviation: {std_nan}")
print(f"Variance:           {var_nan}")

### Minimum, Maximum, Sum, Product

These operations can be computed without sorting.

In [ ]:
min_val = np.nanmin(arr)
max_val = np.nanmax(arr)
sum_val = np.nansum(arr)
prod_val = np.nanprod(arr[:10]) # Use only first 10 rows to avoid numerical overflow

print(f"Minimum: {min_val}")
print(f"Maximum: {max_val}")
print(f"Sum:     {sum_val}")
print(f"Product: {prod_val}")

### Percentiles and Median

Percentiles and medians require sorting. They are therefore slower and less parallelizable.

In [ ]:
p25 = np.nanpercentile(arr, 25)
p50 = np.nanpercentile(arr, 50) # median
p75 = np.nanpercentile(arr, 75)

median = np.nanmedian(arr)

print(f"25th percentile: {p25}")
print(f"50th percentile: {p50}")
print(f"np.nanmedian:    {median}")
print(f"75th percentile: {p75}")


### Aggregation Along Axes

By default, NumPy aggregates over the entire array. You can aggregate per column or per row using axis.

In [ ]:
# Per Column (Feature-wise)
col_mean = np.nanmean(arr, axis=0)
col_std = np.nanstd(arr, axis=0)

print(f"Column means:               {np.round(col_mean, 4)}")
print(f"Column standard deviations: {np.round(col_std, 4)}")

In [ ]:
# Per Row (Sample-wise)
row_mean = np.nanmean(arr, axis=1)
print(f"First 8 row means: {np.round(row_mean[:8], 4)}")